# Image Extension Case Mismatch Fix

Detect and fix image references where the extension case differs between HTML/CSS/JS references and actual files on disk.

In [3]:
import os
import re
from pathlib import Path
from collections import defaultdict

repo_root = Path('.')
print('Repository root:', repo_root.resolve())

Repository root: /workspaces/Atelier-Francis


## Scan Repository for Image References

Search HTML/CSS/JS files for image paths and collect them for validation.

In [ ]:
image_ref_pattern = re.compile(r'(?:src|href)=[\"']([^\"']+\.(?:jpg|jpeg|png|gif|svg|webp|JPG|JPEG|PNG|GIF|SVG|WEBP))[\"']', re.IGNORECASE)
files_to_scan = [p for p in repo_root.rglob('*') if p.suffix.lower() in {'.html', '.css', '.js'}]
refs = []
for p in files_to_scan:
    text = p.read_text(errors='ignore')
    for m in image_ref_pattern.finditer(text):
        ref = m.group(1).strip()
        if ref.startswith(('http://', 'https://', 'data:')):
            continue
        refs.append((p, ref))

print('Scanned files:', len(files_to_scan))
print('Found image references:', len(refs))

In [5]:
canonical_files = defaultdict(list)
for p in repo_root.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.gif', '.svg', '.webp'}:
        canonical_files[str(p.relative_to(repo_root)).lower()].append(p.relative_to(repo_root))

refs = []
for p in files_to_scan:
    text = p.read_text(errors='ignore')
    for m in image_ref_pattern.finditer(text):
        ref = m.group(1).strip()
        if ref.startswith(('http://', 'https://', 'data:')):
            continue
        refs.append((p, ref))

mismatches = []
for file_path, ref in refs:
    ref_path = Path(ref)
    resolved = repo_root / ref_path
    if not resolved.exists():
        key = str(ref_path).lower()
        if key in canonical_files:
            candidates = canonical_files[key]
            mismatches.append((file_path, ref, candidates))
        else:
            mismatches.append((file_path, ref, []))

print('Total mismatches found:', len(mismatches))
for source_file, ref, candidate in mismatches[:50]:
    print(source_file, '->', ref, 'candidates:', candidate)


NameError: name 'files_to_scan' is not defined

In [10]:
from pathlib import Path
import re
from collections import defaultdict

repo_root = Path('.')
image_ref_pattern = re.compile("(?:src|href)=[\"']([^\"']+\.(?:jpg|jpeg|png|gif|svg|webp|JPG|JPEG|PNG|GIF|SVG|WEBP))[\"']", re.IGNORECASE)

files_to_scan = [p for p in repo_root.rglob('*') if p.suffix.lower() in {'.html', '.css', '.js'}]
refs = []
for p in files_to_scan:
    text = p.read_text(errors='ignore')
    for m in image_ref_pattern.finditer(text):
        ref = m.group(1).strip()
        if ref.startswith(('http://', 'https://', 'data:')):
            continue
        refs.append((p, ref))

canonical_files = defaultdict(list)
for p in repo_root.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.gif', '.svg', '.webp'}:
        canonical_files[str(p.relative_to(repo_root)).lower()].append(p.relative_to(repo_root))

mismatches = []
for source_file, ref in refs:
    ref_path = Path(ref)
    resolved = repo_root / ref_path
    if not resolved.exists():
        key = str(ref_path).lower()
        if key in canonical_files:
            candidates = canonical_files[key]
            mismatches.append((source_file, ref, candidates))
        else:
            mismatches.append((source_file, ref, []))

print('Scanned files:', len(files_to_scan))
print('Found image references:', len(refs))
print('Total mismatches found:', len(mismatches))
for source_file, ref, candidate in mismatches[:100]:
    print(source_file, '->', ref, 'candidates:', candidate)


Scanned files: 16
Found image references: 138
Total mismatches found: 3
kurs-modern-hip-hop.html -> Fotos/Bilder von Website/Modern &amp; Hip-Hop/534823d0-0a83-4558-9b25-151937dfe72a.jpg candidates: []
kurs-modern-hip-hop.html -> Fotos/Bilder von Website/Modern &amp; Hip-Hop/234c23f0-c2bd-4758-9f21-aa90a33da220.jpg candidates: []
kurs-modern-hip-hop.html -> Fotos/Bilder von Website/Modern &amp; Hip-Hop/212c6cc3-4ec5-437b-88a5-390f61410097.jpg candidates: []


<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_8072/3670462043.py:6: SyntaxWarning: invalid escape sequence '\.'
  image_ref_pattern = re.compile("(?:src|href)=[\"']([^\"']+\.(?:jpg|jpeg|png|gif|svg|webp|JPG|JPEG|PNG|GIF|SVG|WEBP))[\"']", re.IGNORECASE)


## Detect Uppercase File Extensions

Compare the referenced paths against the filesystem and find mismatches due to extension case differences.

In [4]:
canonical_files = defaultdict(list)
for p in repo_root.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.gif', '.svg', '.webp'}:
        canonical_files[str(p.relative_to(repo_root)).lower()].append(p.relative_to(repo_root))

mismatches = []
for file_path, ref in refs:
    ref_path = Path(ref)
    resolved = repo_root / ref_path
    if not resolved.exists():
        key = str(ref_path).lower()
        if key in canonical_files:
            candidates = canonical_files[key]
            mismatches.append((file_path, ref, candidates))
        else:
            mismatches.append((file_path, ref, []))

print('Total mismatches found:', len(mismatches))
for source_file, ref, candidate in mismatches[:20]:
    print(source_file, '->', ref, 'candidates:', candidate)

NameError: name 'refs' is not defined

## Rename Files and Update References

Rename filesystem image files with uppercase extensions to lowercase, and update references to match.
This section is normally destructive, so we only act if the mismatches are safe to fix.

In [ ]:
def normalize_extension(path: Path) -> Path:
    return path.with_suffix(path.suffix.lower())

changes = []
for file_path, ref, candidates in mismatches:
    if len(candidates) == 1:
        actual = repo_root / candidates[0]
        desired = repo_root / ref
        if actual.exists() and actual != desired and desired.suffix.lower() == actual.suffix.lower():
            changes.append((actual, desired, file_path, ref))

print('Safe rename candidates:', len(changes))
for actual, desired, source_file, ref in changes[:20]:
    print('Rename', actual, '=>', desired, 'from ref', ref, 'in', source_file)

## Validate Image Links

Confirm that all referenced image paths now exist exactly as written.

In [ ]:
broken = []
for p, ref in refs:
    if not (repo_root / ref).exists():
        broken.append((p, ref))

print('Remaining broken references:', len(broken))
for p, ref in broken[:20]:
    print(p, '->', ref)